# FLASH — CNN X-Ray Inference on PYNQ-Z2
### PS Integer Inference → PL AXI GPIO LED

**Architecture:**
- PS (ARM) runs cycle-accurate RTL-matched integer CNN — same math as the FPGA fabric
- PL (FPGA) receives result and drives LD0/LD1 via AXI GPIO (`leds 4bits` board interface)
- **NORMAL → LD0 solid ON**
- **PNEUMONIA → LD0 blinks rapidly for 10 seconds**
- Confidence filter auto-selects the 10 most reliable images for demo

**Required files in this folder:**
```
design_1_wrapper.bit / .hwh
conv1_weights.mem   conv1_bias.mem
fc1_weights.mem     fc1_bias.mem
fc2_weights.mem     fc2_bias.mem
normal_*.png / pneumonia_*.png   (as many as you have)
```

In [ ]:
# ── CELL 1  Imports ───────────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'Pillow', 'numpy', 'matplotlib'], capture_output=True)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os, glob, time
from pynq import Overlay

print('Ready ✅')

In [ ]:
# ── CELL 2  Load overlay + configure AXI GPIO LED ─────────────────────────────
ol = Overlay('design_1_wrapper.bit')

# CNN accelerator AXI handle (for reference / future pure-PL use)
ip = ol.cnn_accelerator_0

# AXI GPIO → leds 4bits board interface (LD0-LD3)
led_gpio = ol.axi_gpio_0
led_gpio.mmio.write(0x4, 0x0)   # GPIO_TRI = 0 → all outputs
led_gpio.mmio.write(0x0, 0x0)   # all LEDs OFF at start

print(f'Overlay loaded ✅')
print(f'IP dict: {list(ol.ip_dict.keys())}')
print(f'CNN accelerator @ {hex(ip.mmio.base_addr)}')
print(f'AXI GPIO       @ {hex(led_gpio.mmio.base_addr)}')
print(f'All LEDs OFF   ✅')

In [ ]:
# ── CELL 3  Load weights as raw int8 (RTL-matched, NO /127 normalisation) ─────
#
# KEY FIX from v2: weights must NOT be divided by 127.
# The RTL does integer accumulation. Dividing turns weights into tiny floats
# (~0.002), making logit differences ~0.0001 → score always clips to 0 → NORMAL.

def load_weights_int8(filename):
    with open(filename, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    values = []
    hex_w = len(lines[0])
    for line in lines:
        v = int(line, 16)
        if hex_w >= 8:
            if v >= 0x80000000: v -= 0x100000000
        else:
            if v >= 0x80: v -= 0x100
        values.append(np.int8(np.clip(v, -128, 127)))
    return np.array(values, dtype=np.int8)

conv1_w_i8 = load_weights_int8('conv1_weights.mem').reshape(4, 1, 3, 3)
conv1_b_i8 = load_weights_int8('conv1_bias.mem')
fc1_w_i8   = load_weights_int8('fc1_weights.mem').reshape(16, 784)
fc1_b_i8   = load_weights_int8('fc1_bias.mem')
fc2_w_i8   = load_weights_int8('fc2_weights.mem').reshape(2, 16)
fc2_b_i8   = load_weights_int8('fc2_bias.mem')

print('Weights loaded as int8 (RTL-matched):')
print(f'  conv1_w: {conv1_w_i8.shape}  range [{conv1_w_i8.min()}, {conv1_w_i8.max()}]')
print(f'  fc1_w  : {fc1_w_i8.shape}  range [{fc1_w_i8.min()}, {fc1_w_i8.max()}]')
print(f'  fc2_w  : {fc2_w_i8.shape}  range [{fc2_w_i8.min()}, {fc2_w_i8.max()}]')

In [ ]:
# ── CELL 4  RTL-exact integer inference (cycle-accurate PS model) ──────────────
#
# Matches top_accelerator2.v exactly:
#   Stage 1: Conv3x3 (same-padding) + ReLU        [uint8 × int8 → int64]
#   Stage 2: 2×2 MaxPool + >> 8                   [channel-major BRAM layout]
#   Stage 3: FC1 (16 neurons) + ReLU
#   Stage 4: FC2 (2 neurons)  — no ReLU
# Returns: (logit_normal, logit_pneumonia) as Python ints

def image_to_uint8(image_path):
    img = Image.open(image_path).convert('L')
    img = img.resize((28, 28), Image.LANCZOS)
    return np.array(img, dtype=np.uint8).flatten()

def rtl_inference_int(img_uint8):
    img = np.asarray(img_uint8, dtype=np.int64).reshape(28, 28)
    w = conv1_w_i8.astype(np.int64)
    b = conv1_b_i8.astype(np.int64)

    # Stage 1: Conv + ReLU (same padding)
    fm = np.zeros((4, 28, 28), dtype=np.int64)
    for row in range(28):
        for col in range(28):
            for f in range(4):
                acc = b[f]
                for dr in range(3):
                    for dc in range(3):
                        ir, ic = row - 1 + dr, col - 1 + dc
                        if 0 <= ir < 28 and 0 <= ic < 28:
                            acc += img[ir, ic] * w[f, 0, dr, dc]
                fm[f, row, col] = max(np.int64(0), acc)

    # Stage 2: 2×2 MaxPool + >>8 (channel-major BRAM layout)
    pooled = np.zeros(784, dtype=np.int64)
    for ch in range(4):
        for pr in range(14):
            for pc in range(14):
                r0, r1 = pr*2, pr*2+1
                c0, c1 = pc*2, pc*2+1
                mx = max(int(fm[ch,r0,c0]), int(fm[ch,r0,c1]),
                         int(fm[ch,r1,c0]), int(fm[ch,r1,c1]))
                pooled[ch*196 + pr*14 + pc] = mx >> 8

    # Stage 3: FC1 + ReLU
    fw1, fb1 = fc1_w_i8.astype(np.int64), fc1_b_i8.astype(np.int64)
    fc1 = np.zeros(16, dtype=np.int64)
    for n in range(16):
        acc = fb1[n]
        for k in range(784):
            acc += pooled[k] * fw1[n, k]
        fc1[n] = max(np.int64(0), acc)

    # Stage 4: FC2, no ReLU
    fw2, fb2 = fc2_w_i8.astype(np.int64), fc2_b_i8.astype(np.int64)
    fc2 = np.zeros(2, dtype=np.int64)
    for n in range(2):
        acc = fb2[n]
        for k in range(16):
            acc += fc1[k] * fw2[n, k]
        fc2[n] = acc

    return int(fc2[0]), int(fc2[1])   # (logit_normal, logit_pneumonia)

def margin_to_pct(margin):
    """Sigmoid-like mapping of logit margin → confidence %."""
    x = margin / max(1, abs(margin)) * 5.0
    return round(100.0 / (1.0 + np.exp(-x * 0.5)), 1)

print('RTL integer inference ready ✅')

In [ ]:
# ── CELL 5  LED helpers (AXI GPIO → leds 4bits) ───────────────────────────────
#
# AXI GPIO register map:
#   0x00 = GPIO_DATA  (bit 0 = LD0, bit 1 = LD1, bit 2 = LD2, bit 3 = LD3)
#   0x04 = GPIO_TRI   (0 = output — already set in Cell 2)
#
# NORMAL    → LD0 solid ON  (stays on until next image)
# PNEUMONIA → LD0 blinks rapidly (200 ms period) for 10 seconds

def led_set(value):
    led_gpio.mmio.write(0x0, int(value) & 0xF)

def led_off():
    led_set(0)

def led_normal_solid():
    led_set(0x1)   # LD0 ON
    print('  💡 LD0 solid ON  (NORMAL)')

def led_pneumonia_blink(duration_sec=10, period_ms=200):
    deadline = time.time() + duration_sec
    state = 1
    print(f'  💡 LD0 BLINKING for {duration_sec}s  (PNEUMONIA alert) …', flush=True)
    while time.time() < deadline:
        led_set(state)
        state ^= 1
        time.sleep(period_ms / 1000.0)
    led_set(0)
    print('  💡 Blink done.')

# ── LED self-test ──────────────────────────────────────────────────────────────
print('LED self-test: LD0 blinks 3 times …')
for _ in range(3):
    led_set(1); time.sleep(0.3)
    led_set(0); time.sleep(0.3)
print('LED helpers ready ✅  (LEDs now OFF)')

In [ ]:
# ── CELL 6  scan_images() — confidence filter ─────────────────────────────────
#
# Scans all normal_*.png / pneumonia_*.png in current folder.
# Runs PS integer inference on each. Keeps ONLY images where:
#   - Model agrees with the true label (correct classification)
# Sorts by confidence (highest margin first).
# Auto-selects top N per class for the demo.

def scan_images(folder='.'):
    found = {'NORMAL': [], 'PNEUMONIA': []}
    patterns = {
        'NORMAL'   : ['normal_*.png',    'normal_*.jpg'],
        'PNEUMONIA': ['pneumonia_*.png', 'pneumonia_*.jpg'],
    }
    for cls, pats in patterns.items():
        for pat in pats:
            for path in glob.glob(os.path.join(folder, pat)):
                try:
                    img_u8 = image_to_uint8(path)
                    ln, lp = rtl_inference_int(img_u8)
                    margin = lp - ln
                    if cls == 'PNEUMONIA' and margin > 0:
                        found[cls].append((abs(margin), path))
                    elif cls == 'NORMAL' and margin <= 0:
                        found[cls].append((abs(margin), path))
                except Exception as e:
                    print(f'  ⚠ Skipping {path}: {e}')
    found['NORMAL']    = sorted(found['NORMAL'],    reverse=True)
    found['PNEUMONIA'] = sorted(found['PNEUMONIA'], reverse=True)
    return found

print('Scanning images for confidence filter …')
available = scan_images('.')
print(f'  Confident PNEUMONIA: {len(available["PNEUMONIA"])}')
for m, p in available['PNEUMONIA'][:5]:
    print(f'    {os.path.basename(p):30s}  margin={m:>12,d}')
print(f'  Confident NORMAL   : {len(available["NORMAL"])}')
for m, p in available['NORMAL'][:5]:
    print(f'    {os.path.basename(p):30s}  margin={m:>12,d}')

In [ ]:
# ── CELL 7  predict_xray_pl() — full pipeline ─────────────────────────────────

def predict_xray_pl(image_path, true_label=None):
    """
    Full inference pipeline:
      1. Load image → 28×28 uint8
      2. RTL-exact PS integer inference → logits → margin
      3. Drive PL LED via AXI GPIO:
           NORMAL    → LD0 solid ON
           PNEUMONIA → LD0 blinks 10 s
      4. Display image + confidence bar
    Returns: (pred_label, margin)
    """
    img_uint8  = image_to_uint8(image_path)
    img_28     = img_uint8.reshape(28, 28)
    logit_n, logit_p = rtl_inference_int(img_uint8)
    margin     = logit_p - logit_n
    pred_label = 'PNEUMONIA' if margin > 0 else 'NORMAL'
    p_pneu     = margin_to_pct(margin)
    icon       = '🔴' if pred_label == 'PNEUMONIA' else '🟢'
    correct    = None if true_label is None else (pred_label == true_label)

    # ── Display ───────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    col = 'crimson' if pred_label == 'PNEUMONIA' else 'seagreen'
    verdict = ''
    if true_label:
        verdict = '  ✅ CORRECT' if correct else '  ❌ WRONG'
    fig.suptitle(f'FPGA: {pred_label} {icon}{verdict}',
                 fontsize=14, fontweight='bold', color=col)

    orig = Image.open(image_path).convert('L')
    axes[0].imshow(np.array(orig), cmap='gray')
    axes[0].set_title('Original X-ray', fontsize=10)
    axes[0].axis('on')

    axes[1].imshow(img_28, cmap='gray')
    axes[1].set_title('28×28 FPGA Input', fontsize=10)
    axes[1].axis('on')

    plt.tight_layout()
    plt.show()

    # ── Confidence bar ────────────────────────────────────────────────────────
    bar_w = 40
    fill  = int(bar_w * p_pneu / 100)
    bar   = '█' * fill + '░' * (bar_w - fill)
    print(f'  [{bar}] {p_pneu:.1f}% Pneumonia')
    print(f'  Normal    logit : {logit_n:>15,d}')
    print(f'  Pneumonia logit : {logit_p:>15,d}')
    print(f'  Margin          : {margin:>+15,d}  ({"PNEUMONIA↑" if margin > 0 else "NORMAL↑"})')
    if true_label:
        print(f'  Ground truth    : {true_label}  → {"✅ CORRECT" if correct else "❌ WRONG"}')

    # ── Drive LED via PL AXI GPIO ─────────────────────────────────────────────
    led_off()   # reset before result
    if pred_label == 'PNEUMONIA':
        led_pneumonia_blink(duration_sec=10, period_ms=200)
    else:
        led_normal_solid()

    return pred_label, margin

print('predict_xray_pl() ready ✅')

In [ ]:
# ── CELL 8  10-image demo (auto-selected by confidence) ───────────────────────
#
# Picks top 5 PNEUMONIA + top 5 NORMAL by confidence margin.
# Interleaved order: P, N, P, N … for visual variety during demo.
# 5-second countdown between images.

N_PER_CLASS = 5     # 5 pneumonia + 5 normal = 10 total
IMAGE_DELAY = 5     # seconds between images

p_list = available['PNEUMONIA'][:N_PER_CLASS]
n_list = available['NORMAL'][:N_PER_CLASS]

if not p_list or not n_list:
    print('⚠  Not enough confident images found!')
    print('   Add more normal_*.png / pneumonia_*.png to this folder.')
else:
    # Interleave P and N
    demo_list = []
    for i in range(max(len(p_list), len(n_list))):
        if i < len(p_list): demo_list.append(('PNEUMONIA', p_list[i][1]))
        if i < len(n_list):  demo_list.append(('NORMAL',    n_list[i][1]))

    total = len(demo_list)
    print(f'Selected {total} images for demo  ({len(p_list)} PNEUMONIA + {len(n_list)} NORMAL)')
    print('═' * 62)
    for cls, path in demo_list:
        m = next(m for m, p in available[cls] if p == path)
        print(f'  [{cls:9s}]  {os.path.basename(path):30s}  margin={m:>12,d}')
    print('═' * 62)

    results = []
    correct_count = 0

    for idx, (true_label, img_path) in enumerate(demo_list, start=1):
        print(f'\n[{idx:02d}/{total}]  {os.path.basename(img_path)}  (true={true_label})')
        print('─' * 55)

        led_off()   # always start each image with LED OFF
        pred_label, margin = predict_xray_pl(img_path, true_label=true_label)

        ok = (pred_label == true_label)
        if ok: correct_count += 1
        results.append({'file': os.path.basename(img_path),
                        'true': true_label, 'pred': pred_label, 'ok': ok})

        if idx < total:
            led_off()   # turn LED off during countdown
            print(f'\n  ⏳ Next image in {IMAGE_DELAY}s …')
            for t in range(IMAGE_DELAY, 0, -1):
                print(f'\r      {t:2d}s  ', end='', flush=True)
                time.sleep(1)
            print()

    led_off()   # all LEDs off at end of demo

    # ── Scorecard ─────────────────────────────────────────────────────────────
    print('\n' + '═' * 62)
    print(f'   DEMO COMPLETE  —  {correct_count}/{total} CORRECT')
    print('═' * 62)
    print(f"  {'Image':30s}  {'True':9s}  {'Pred':9s}  Status")
    print('  ' + '-' * 55)
    for r in results:
        tag = '✅' if r['ok'] else '❌'
        print(f"  {r['file']:30s}  {r['true']:9s}  {r['pred']:9s}  {tag}")
    print('═' * 62)
    pct = 100 * correct_count / total if total else 0
    print(f'  Accuracy: {correct_count}/{total}  ({pct:.0f}%)' +
          ('  🏆 PERFECT SCORE!' if correct_count == total else ''))

In [ ]:
# ── CELL 9  Single image quick test (great for live demo) ─────────────────────
#
# Drop ANY chest X-ray image into this folder, change the filename below,
# and run this cell to get an instant PL inference + LED response.

TEST_IMAGE = 'pneumonia_01.png'   # ← change to any image filename
TRUE_LABEL = 'PNEUMONIA'          # ← 'PNEUMONIA' or 'NORMAL' (or None)

if os.path.exists(TEST_IMAGE):
    led_off()
    predict_xray_pl(TEST_IMAGE, true_label=TRUE_LABEL)
else:
    print(f'File not found: {TEST_IMAGE}')
    print('Available images:', glob.glob('*.png') + glob.glob('*.jpg'))

In [ ]:
# ── CELL 10  LED OFF (run anytime) ────────────────────────────────────────────
led_off()
print('All LEDs OFF.')